# reflect_distill_trec21 — error-reflective prompt optimization (dev = TREC21)

Design §5.2-5.4 / §8e. The scored labeler's ceiling is **false positives** (non-eligible trials scored
high). This mines those errors, distills a guidance blob, appends it, and measures the lift —
**base labeler vs base+guidance, both Claude, both on TREC21** (clean: neither trained on it; the clf
contamination that broke labeler-vs-clf doesn't touch labeler-vs-labeler).

Split TREC21: mine on **train**, gate/report on **dev**. TREC22 untouched (spent once, later).
Base labeler is fixed vs §8e: `max_tokens=128` (kills the 13-eligible parse leak).

## Setup

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q anthropic nest_asyncio pandas tqdm datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, time, re, asyncio, random, collections
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval, full_blob, ndcg_at_k
cfg = ExperimentConfig(data_root=DATA_ROOT)
SPLIT = 'trec21'
CLAUDE_MODEL = 'claude-opus-4-8'
PRICE_IN, PRICE_OUT = 5.0, 25.0
LABEL_W = 100
PROBE_N = 30            # total TREC21 topics used (None = all 75)
TRAIN_FRAC = 0.67       # rest is dev
MAX_CONCURRENT = 15
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    assert os.environ.get('ANTHROPIC_API_KEY'), 'set ANTHROPIC_API_KEY'
usage_tot = [0, 0]      # (in, out) across every Claude call
print('key set:', bool(os.environ.get('ANTHROPIC_API_KEY')))

In [ ]:
corpus_ids, corpus_fields = load_corpus(cfg)
id2fields = dict(zip(corpus_ids, corpus_fields))
ev = load_eval(cfg, [SPLIT])[SPLIT]
rel, topic2text = ev['rel_dict'], ev['topic2text']
topics = [t for t in rel if t in topic2text]
if PROBE_N: topics = topics[:PROBE_N]
n_tr = int(round(TRAIN_FRAC * len(topics)))
train_topics, dev_topics = topics[:n_tr], topics[n_tr:]
pool = json.load(open(cfg.path('data/pool_R.json')))[SPLIT]
ce = np.load(cfg.ce_cache_path('clf'), allow_pickle=True)['d'].item()
clf_scores = {t: {d: ce[(SPLIT, t, d)][0] for d in pool[t] if (SPLIT, t, d) in ce} for t in topics}
clf_order  = {t: sorted(pool[t], key=lambda d: clf_scores[t].get(d, -1.0), reverse=True) for t in topics}
txt = lambda d: full_blob(id2fields[d], cfg)
print(f'train={len(train_topics)} dev={len(dev_topics)} topics | base pass = {len(topics)*LABEL_W} calls')

## Base scored labeler (fixed) + reusable resumable pass

In [ ]:
def make_prompt(topic_text, doc_text, guidance=''):
    p = (f'Patient case:\n{topic_text}\n\nClinical trial:\n{doc_text}\n\n'
         'Give a single relevance score from 0 to 100:\n'
         '90-100 = clearly eligible: trial targets the patient\'s condition AND the patient meets its criteria.\n'
         '30-60  = on-topic but the patient appears excluded by the criteria.\n'
         '0-15   = not relevant: trial does not target the patient\'s condition.\n'
         'Use the full range to reflect confidence.\n')
    if guidance:
        p += f'\nScoring guidance (learned from prior errors):\n{guidance}\n'
    p += '\nRespond with only: SCORE: N'
    return p
def parse_score(t):
    m = re.search(r'SCORE:\s*(\d{1,3})', t)
    if m: return max(0, min(100, int(m.group(1))))
    m = re.findall(r'\d{1,3}', t); return max(0, min(100, int(m[-1]))) if m else None

In [ ]:
import anthropic, nest_asyncio
nest_asyncio.apply()
aclient = anthropic.AsyncAnthropic()
async def score_one(topic_text, doc_text, guidance, sem):
    kw = dict(model=CLAUDE_MODEL, system='You are a clinical trial relevance assessor.',
              thinking={'type': 'disabled'}, max_tokens=128,
              messages=[{'role': 'user', 'content': make_prompt(topic_text, doc_text, guidance)}])
    async with sem:
        for a in range(4):
            try:
                m = await aclient.messages.create(**kw)
                s = parse_score(''.join(b.text for b in m.content if b.type == 'text'))
                return (s if s is not None else 0, m.usage.input_tokens, m.usage.output_tokens, s is not None)
            except Exception as e:
                if a == 3: return 0, 0, 0, False
                await asyncio.sleep(2 ** a)

def run_score_pass(tlist, guidance, tag):
    path = cfg.path(f'data/scores_{tag}_{SPLIT}.jsonl'); sc = {}; done = set()
    if os.path.exists(path):
        for l in open(path):
            r = json.loads(l); sc[(r['topic_id'], r['doc_id'])] = r['score']; done.add(r['topic_id'])
            usage_tot[0] += r['in_tok']; usage_tot[1] += r['out_tok']
    with open(path, 'a') as f:
        for t in tqdm(tlist, desc=f'score:{tag}'):
            if t in done: continue
            sem = asyncio.Semaphore(MAX_CONCURRENT)
            res = asyncio.run(asyncio.gather(*[score_one(topic2text[t], txt(d), guidance, sem) for d in clf_order[t][:LABEL_W]]))
            for d, (s, i, o, ok) in zip(clf_order[t][:LABEL_W], res):
                sc[(t, d)] = s; usage_tot[0] += i; usage_tot[1] += o
                f.write(json.dumps({'topic_id': t, 'doc_id': d, 'score': s, 'in_tok': i, 'out_tok': o, 'ok': ok}) + '\n')
            f.flush()
    return sc

In [ ]:
# Base pass over train+dev (guidance='')
scores_base = run_score_pass(topics, '', 'base')
def ndcg_of(scmap, tset):
    def rank(t):
        top = clf_order[t][:LABEL_W]
        return sorted(top, key=lambda d: (scmap.get((t, d), 0), clf_scores[t].get(d, 0)), reverse=True) + clf_order[t][LABEL_W:]
    return float(np.mean([ndcg_at_k(rank(t), rel[t]) for t in tset]))
print(f'base NDCG@10  dev={ndcg_of(scores_base, dev_topics):.4f}  train={ndcg_of(scores_base, train_topics):.4f}')

## Reflect on TRAIN errors (weighted to false positives)

FP = non-eligible (gold 0/1) scored ≥60; FN = eligible (gold 2) scored <60. Opus (reasoning on — this
is offline data-gen) diagnoses each and proposes a reusable rule.

In [ ]:
FP_T, FN_T, N_FP, N_FN = 60, 60, 60, 20
fps = [(t, d, scores_base[(t, d)], rel[t].get(d, 0)) for t in train_topics for d in clf_order[t][:LABEL_W]
       if (t, d) in scores_base and rel[t].get(d, 0) <= 1 and scores_base[(t, d)] >= FP_T]
fns = [(t, d, scores_base[(t, d)], rel[t].get(d, 0)) for t in train_topics for d in clf_order[t][:LABEL_W]
       if (t, d) in scores_base and rel[t].get(d, 0) == 2 and scores_base[(t, d)] < FN_T]
rng = random.Random(0)
err_sample = rng.sample(fps, min(N_FP, len(fps))) + rng.sample(fns, min(N_FN, len(fns)))
print(f'train errors: {len(fps)} FP, {len(fns)} FN -> reflecting on {len(err_sample)}')

In [ ]:
GOLD_DESC = {2: 'Eligible (should score ~90-100)', 1: 'Excluded / on-topic but patient ineligible (should score ~30-60)',
             0: 'Not relevant (should score ~0-15)'}
async def reflect_one(t, d, s, g, sem):
    prompt = (f'A relevance scorer gave this trial {s}/100 for this patient, but the correct label is: '
              f'{GOLD_DESC[g]}.\n\nPatient case:\n{topic2text[t]}\n\nClinical trial:\n{txt(d)}\n\n'
              'In 1-2 sentences: why did the scorer get this wrong, and what general, reusable scoring '
              'rule would prevent this class of error? State the rule imperatively.')
    async with sem:
        for a in range(4):
            try:
                m = await aclient.messages.create(model=CLAUDE_MODEL,
                        system='You improve a clinical-trial relevance scorer by diagnosing its mistakes.',
                        thinking={'type': 'adaptive'}, output_config={'effort': 'medium'}, max_tokens=1024,
                        messages=[{'role': 'user', 'content': prompt}])
                usage_tot[0] += m.usage.input_tokens; usage_tot[1] += m.usage.output_tokens
                return ''.join(b.text for b in m.content if b.type == 'text').strip()
            except Exception as e:
                if a == 3: return ''
                await asyncio.sleep(2 ** a)
sem = asyncio.Semaphore(MAX_CONCURRENT)
diagnoses = asyncio.run(asyncio.gather(*[reflect_one(*e, sem) for e in err_sample]))
diagnoses = [d for d in diagnoses if d]
json.dump(diagnoses, open(cfg.path(f'data/reflections_{SPLIT}.json'), 'w'))
print(f'{len(diagnoses)} diagnoses. sample:\n-', diagnoses[0][:300])

## Distill → one guidance blob (dev-gated)

In [ ]:
numbered = '\n'.join(f'{i+1}. {d}' for i, d in enumerate(diagnoses))
distill_prompt = ('Below are diagnoses of a clinical-trial relevance scorer\'s errors (mostly false '
    'positives — non-eligible trials scored too high).\n\n' + numbered + '\n\n'
    'Synthesize these into at most 5 short, operational scoring rules that would fix the most common '
    'errors, especially false positives. Each rule must be concrete enough to change a score. '
    'Output ONLY the bulleted rules, <=150 words total.')
m = asyncio.run(aclient.messages.create(model=CLAUDE_MODEL,
        system='You distill error diagnoses into a compact scoring-rubric addendum.',
        thinking={'type': 'adaptive'}, output_config={'effort': 'high'}, max_tokens=2048,
        messages=[{'role': 'user', 'content': distill_prompt}]))
usage_tot[0] += m.usage.input_tokens; usage_tot[1] += m.usage.output_tokens
guidance = ''.join(b.text for b in m.content if b.type == 'text').strip()
open(cfg.path(f'data/guidance_{SPLIT}.txt'), 'w').write(guidance)
print(guidance)

## Guided pass on DEV + the ablation

In [ ]:
scores_guided = run_score_pass(dev_topics, guidance, 'guided')
base_dev   = ndcg_of(scores_base, dev_topics)
guided_dev = ndcg_of(scores_guided, dev_topics)
clf_dev    = float(np.mean([ndcg_at_k(clf_order[t], rel[t]) for t in dev_topics]))
orc_dev    = float(np.mean([ndcg_at_k(sorted(clf_order[t][:LABEL_W], key=lambda d: rel[t].get(d,0), reverse=True)+clf_order[t][LABEL_W:], rel[t]) for t in dev_topics]))
tot_cost = (usage_tot[0]*PRICE_IN + usage_tot[1]*PRICE_OUT)/1e6
print(f'DEV NDCG@10   base={base_dev:.4f}   base+guidance={guided_dev:.4f}   lift={guided_dev-base_dev:+.4f}')
print(f'reference     clf(contaminated)={clf_dev:.4f}   oracle={orc_dev:.4f}')
print(f'total spend so far: ${tot_cost:.2f}')
pd.DataFrame([{'base_dev': round(base_dev,4), 'guided_dev': round(guided_dev,4),
               'lift': round(guided_dev-base_dev,4), 'oracle_dev': round(orc_dev,4),
               'total_$': round(tot_cost,2)}]).to_csv(cfg.path(f'data/reflect_result_{SPLIT}.csv'), index=False)

## Read
- **guided > base by a real margin on dev** → reflection works; the guidance blob is the artifact.
  Freeze it, then one TREC22 run scores base vs base+guidance for the held-out headline.
- **lift ~0 or negative** → distilled guidance doesn't transfer; clean null for the negative-results
  paper (the honest, publishable outcome of the method).
- The blob is saved to `data/guidance_trec21.txt` — inspect whether its rules are operational
  (calibrated-inference / strict-condition-match) or vague. Vague + no lift = distillation collapse (§7).